# Cohort external-test selection

## Introduction

This notebook selects three whole kidney images for external testing before the training cohort is constructed. The training cohort is every complete source image not selected here; the legacy merged `kidney` directory is excluded.

## Assumptions

The annotation CSV represents the available METASPACE molecular population, and the `x<int>_y<int>` columns in `pixel_intensities.csv` represent pixels. Selection must preserve every molecule label in the remaining training images.

## Notation

For an image $i$, $P_i$ is its pixel count, $A_i$ its annotation-record count and $C_i$ its number of unique `formula|adduct` labels.

## Reproducible metadata scan

### Methodology

The scan reads every annotation CSV and only the header of every intensity CSV. Candidates are inside the 20th--80th percentile for log(1 + P_i), log(1 + A_i) and log(1 + C_i). It enumerates triples, rejects every triple that removes a label from training, and chooses the lowest robust distance to the cohort median.

### Theoretical description

The lossless constraint is: union of labels outside held-out set H equals the union across the whole cohort. Consequently a held-out image cannot be the only training source of any molecular class.

### Implementation description

The reusable implementation is `analysis.autoencoder.experiments.cohort_selection`; it writes canonical CSV tables and provenance. The notebook only loads and presents those artifacts.

### Figure description

No plot is required for this preflight decision; the numeric distribution and selected rows are displayed below.

### Remarks

The scan is metadata-only and does not materialize spectra or an intensity matrix. It is therefore not a throughput benchmark for training.

### Notes


In [1]:
from pathlib import Path

import pandas as pd

from msi_autoencoder_wrapper.analysis.autoencoder.experiments.cohort_selection import (
    load_cohort_selection_results,
)

RESULTS_DIRECTORY = Path('part_0_01_cohort_selection_results')
results = load_cohort_selection_results(RESULTS_DIRECTORY)
images = results['images']
heldout = results['heldout']
coverage = results['label_coverage']


2026-09-20 17:04:23,141 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.
2026-09-20 17:04:23,144 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.


In [2]:
display(images[['pixel_count', 'annotation_records', 'unique_label_count']].describe(percentiles=[0.2, 0.5, 0.8]))
display(heldout)
assert (coverage['training_image_count'] > 0).all(), 'A held-out selection removed a training label.'
print(f"Training label coverage: {len(coverage)}/{len(coverage)} labels retained.")


,pixel_count,annotation_records,unique_label_count
count,50.000000,50.000000,50.000000
mean,30134.160000,42.940000,29.920000
std,25945.766679,78.275132,50.516855
min,5025.000000,4.000000,3.000000
20%,8241.800000,8.800000,7.000000
50%,11786.500000,14.500000,10.000000
80%,55146.600000,40.600000,32.200000
max,83704.000000,467.000000,298.000000


,image_key,dataset_name,pixel_count,annotation_records,unique_label_count,is_middle_candidate,is_heldout,pixel_count_percentile,annotation_records_percentile,unique_label_count_percentile
0,2024-02-20_01h45m20s,d3-2023-2,10994,11,7,True,True,0.48,0.33,0.24
1,2024-02-20_01h46m58s,d7-2020-2,9888,13,9,True,True,0.36,0.44,0.41
2,2024-02-20_01h54m41s,d28-2006-2,8566,16,13,True,True,0.30,0.58,0.60


Training label coverage: 743/743 labels retained.


In [3]:
print(f"Total datasets: {len(images)}")

display(
    images[
        [
            "pixel_count",
            "annotation_records",
            "unique_label_count",
        ]
    ]
    .sort_values("pixel_count", ascending=False)
)

Total datasets: 50


,pixel_count,annotation_records,unique_label_count
29,83704,110,97
37,81071,209,138
36,81071,209,138
42,74880,142,92
41,71446,33,25
28,70784,28,20
40,69127,11,9
39,63195,8,8
48,56701,43,28
32,55793,10,9


In [4]:
display(
    coverage["training_image_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_training_images")
    .to_frame("label_count")
)

print(
    "Labels occurring in exactly one training image:",
    (coverage["training_image_count"] == 1).sum()
)

,label_count
number_of_training_images,
1,475
2,131
3,44
4,33
5,18
6,7
7,7
8,6
9,6


Labels occurring in exactly one training image: 475
